# MODIS + VIIRS harmonization — Steps 2 to 5 (DRAFT)

**Status: draft written before the real files arrived.**
Column names here are what I *expect* from FIRMS. They are not verified yet.
Cell 4 checks them and stops with a clear error if they are wrong.

Run one cell at a time. Read the "Check" note under each step before moving on.

**AI use:** the code in this notebook was drafted with Claude (Anthropic), an AI tool. Declare this on the project page.

## Setup

In [ ]:
import os, glob, zipfile, shutil
import pandas as pd

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

## Upload the FIRMS zips (Colab only)

Pick both zip files when the button appears. Big files upload slowly. Be patient.

In [ ]:
from google.colab import files as colab_files

uploaded = colab_files.upload()
for name in uploaded:
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(name).extractall(DATA_DIR)
    elif name.lower().endswith(".csv"):
        shutil.move(name, os.path.join(DATA_DIR, name))
print(os.listdir(DATA_DIR))

## Step 2 — Inspect the files

**What and why.** We read only 5 rows of each file. We look at the real column names before writing any logic. Guessing names is the fastest way to get silent wrong answers.

**Check.** Write down the column list for MODIS and for VIIRS. They are not identical.

**Common mistake.** Opening the file in Excel. It cuts off after about 1 million rows.

In [ ]:
csvs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print(len(csvs), "CSV files found")
for f in csvs:
    peek = pd.read_csv(f, nrows=5)
    print()
    print(os.path.basename(f), "-", round(os.path.getsize(f) / 1e6, 1), "MB")
    print(list(peek.columns))
    print(peek.to_string())

### Column check

This cell stops if an expected column is missing. If it stops, do not edit around it. Tell Claude the column list from the cell above.

In [ ]:
NEEDED = ["latitude", "longitude", "acq_date", "confidence", "instrument"]

files = {"MODIS": [], "VIIRS": []}
for f in csvs:
    peek = pd.read_csv(f, nrows=5)
    if len(peek) == 0:
        print("Skipping empty file:", os.path.basename(f))
        continue
    missing = [c for c in NEEDED if c not in peek.columns]
    if missing:
        raise ValueError(f"{os.path.basename(f)} is missing columns {missing}. Columns found: {list(peek.columns)}")
    inst = str(peek["instrument"].iloc[0]).strip().upper()
    if inst not in files:
        raise ValueError(f"{os.path.basename(f)} has unexpected instrument value: {inst}")
    files[inst].append(f)

for k, v in files.items():
    print(k, [os.path.basename(x) for x in v])

## Load the data — small sample first

`SAMPLE = True` reads only the first 50,000 rows of each file. That is enough to test the code.
The sample is the *earliest* rows only, so the numbers mean nothing yet.

When every cell below runs cleanly, set `SAMPLE = False` and run everything again.

In [ ]:
SAMPLE = True
NROWS = 50_000 if SAMPLE else None

def load(sensor):
    parts = []
    for f in files[sensor]:
        d = pd.read_csv(f, nrows=NROWS, dtype={"confidence": str})
        d["source_file"] = os.path.basename(f)
        parts.append(d)
    out = pd.concat(parts, ignore_index=True)
    out["date"] = pd.to_datetime(out["acq_date"])
    return out

raw = {s: load(s) for s in ["MODIS", "VIIRS"]}
for s, d in raw.items():
    print(s, len(d), "rows,", d["date"].min().date(), "to", d["date"].max().date())
    print(d.groupby("source_file")["date"].agg(["min", "max", "count"]))

### Look at the values we will filter on

**Check.**
- MODIS `confidence` should be numbers 0 to 100.
- VIIRS `confidence` should be letters (low / nominal / high, probably `l`, `n`, `h`).
- `type` may exist only in the archive files, not the NRT files.
- `satellite` shows which satellite made each detection.

If any of these look different, stop and tell Claude.

In [ ]:
for s, d in raw.items():
    print("=====", s, "=====")
    print("confidence:", d["confidence"].value_counts(dropna=False).head(12).to_dict())
    for col in ["type", "satellite", "daynight", "version"]:
        if col in d.columns:
            print(col + ":", d[col].value_counts(dropna=False).to_dict())
        else:
            print(col + ": (no such column)")

## Step 3 — Clean

**What and why.** Three filters, applied the same way to both sensors:

1. **Sabah box.** Keep lat 4.0 to 7.5, lon 115.3 to 119.3. This is an approximate rectangle. It also catches Brunei and bits of Sarawak and North Kalimantan. That goes in the limits section.
2. **Burning type.** If a `type` column exists, keep type 0 (presumed vegetation fire). Drop 1 volcano, 2 static land source, 3 offshore (gas flares). Rows with no type (NRT files) are kept. Source: MODIS C6/C6.1 user guide, Table 10.
3. **Confidence rule: drop low confidence.** MODIS: keep confidence >= 30. VIIRS: keep nominal and high. Source: MODIS C6/C6.1 user guide, Table 11 (low is 0 to 30) and section 5.1.3.

**Check.** Read the printed row counts. Each filter should remove some rows, not all rows.

**Common mistake.** Using a different confidence rule for each sensor without writing it down. Judges will ask.

In [ ]:
LAT_MIN, LAT_MAX = 4.0, 7.5
LON_MIN, LON_MAX = 115.3, 119.3
MODIS_MIN_CONF = 30

def clean(df, sensor):
    print("=====", sensor, "=====")
    print("start:           ", len(df))
    df = df[df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX)]
    print("inside Sabah box:", len(df))
    if "type" in df.columns:
        df = df[df["type"].isna() | (df["type"] == 0)]
        print("vegetation type: ", len(df))
    else:
        print("no type column, skipped")
    if sensor == "MODIS":
        keep = pd.to_numeric(df["confidence"], errors="coerce") >= MODIS_MIN_CONF
    else:
        keep = df["confidence"].str.strip().str.lower().str[0].isin(["n", "h"])
    df = df[keep]
    print("not low conf:    ", len(df))
    return df

cleaned = {s: clean(raw[s], s) for s in ["MODIS", "VIIRS"]}

has_type = {s: "type" in raw[s].columns for s in raw}
if has_type["MODIS"] != has_type["VIIRS"]:
    print()
    print("WARNING: only one sensor has a type column:", has_type)
    print("The type filter is then uneven between sensors. Tell Claude before going on.")

## Step 4 — Count hotspots per month per sensor

**What and why.** One row per month, one column per sensor. A month with no fires must be **0**. A month when the sensor did not exist must be **empty (NaN)**. Those are different things.

**Check.** MODIS should start in late 2000. VIIRS should be empty before 2012.

**Common mistake.** Letting zero-fire months vanish from the table. Then medians and ratios are wrong.

In [ ]:
def monthly(clean_df, raw_df):
    # month range comes from the RAW Borneo data, so quiet Sabah months still appear as 0
    full = pd.period_range(raw_df["date"].min().to_period("M"), raw_df["date"].max().to_period("M"), freq="M")
    counts = clean_df["date"].dt.to_period("M").value_counts()
    return counts.reindex(full, fill_value=0)

table = pd.DataFrame({
    "modis_raw": monthly(cleaned["MODIS"], raw["MODIS"]),
    "viirs": monthly(cleaned["VIIRS"], raw["VIIRS"]),
})
# MODIS split by satellite (T = Terra, A = Aqua). Used to see the Terra-only early years.
if "satellite" in cleaned["MODIS"].columns:
    sat = cleaned["MODIS"]["satellite"].astype(str).str.strip().str.upper().str[0]
    for letter, col in [("T", "modis_terra"), ("A", "modis_aqua")]:
        table[col] = monthly(cleaned["MODIS"][sat == letter], raw["MODIS"])
table.index.name = "period"
table = table.reset_index()
table["year"] = table["period"].dt.year
table["month"] = table["period"].dt.month
extra = [c for c in ["modis_terra", "modis_aqua"] if c in table.columns]
table = table[["period", "year", "month", "modis_raw", "viirs"] + extra]
print(table.head(8).to_string())
print(table.tail(8).to_string())

## Step 5 — Find the overlap years, and check MODIS

**What and why.** Ratios can only be learned from months when both sensors worked. We also check which MODIS satellite contributed each year. MODIS is two satellites (Terra and Aqua). If one is missing in some years, MODIS itself is not consistent.

**Check.**
- Overlap should start Feb 2012. January 2012 is partial (VIIRS began on the 20th), so we drop it.
- In the satellite table, expect Terra only before mid-2002. That is a second break to mention to judges.
- Look at the last MODIS years. If counts collapse for one satellite, say so.

**Common mistake.** Including the partial first month in the ratio. It makes VIIRS look too low.

In [ ]:
VIIRS_FIRST_FULL_MONTH = pd.Period("2012-02", "M")

overlap = table.dropna(subset=["modis_raw", "viirs"])
overlap = overlap[overlap["period"] >= VIIRS_FIRST_FULL_MONTH]
if len(overlap) == 0:
    print("No overlap months. Expected while SAMPLE = True. Set SAMPLE = False and rerun.")
else:
    print("Overlap:", overlap["period"].min(), "to", overlap["period"].max(), "-", len(overlap), "months")

if "modis_aqua" in table.columns:
    print("MODIS detections per year, by satellite (expect Aqua = 0 before mid-2002):")
    print(table.groupby("year")[["modis_terra", "modis_aqua"]].sum().to_string())
print("Last MODIS detection in the raw file:", raw["MODIS"]["date"].max().date())
print("Last VIIRS detection in the raw file:", raw["VIIRS"]["date"].max().date())

## Save

Only trust this file when `SAMPLE = False`.

In [ ]:
name = "monthly_counts_SAMPLE.csv" if SAMPLE else "monthly_counts.csv"
table.drop(columns="period").to_csv(name, index=False)
print("saved", name, "-", len(table), "rows")